# Skin T cells — TCR clonality + inferCNV malignancy (combined atlas)

Per-cell malignancy on the combined skin-T object `skin_T_annotated.h5ad` (studies: li2024, chennareddy, brunner, gaydosik, herrera, rindler) from two orthogonal signals:

- **TCR clonality** — recompute the dominant-clone malignancy uniformly (top clone ≥5% of the donor's TCR cells **and** ≥2× the 2nd clone).
- **inferCNV** — per-donor copy-number score vs a same-lineage T-cell reference, called **four ways** (2×2 grid below).

Heavy helpers live in `skin_T_cnv_helpers.py` (imported as `C`); this notebook just orchestrates.

### The 4 inferCNV methods — `{base call} × {smoothing}`

| key | obs column | base call | smoothing |
|---|---|---|---|
| `strat1_cell_gmm` | `cnv_malignant` | per-cell GMM on the focal score | none |
| `strat2_cell_mrvi` | `cnv_malig_leiden_mrvi` | per-cell GMM | high-res MRVI-Leiden majority vote |
| `strat3_cnv_cluster` | `cnv_malig_cluster` | per-cnv-cluster GMM | none |
| `strat4_cnv_cluster_mrvi` | `cnv_malig_cluster_mrvi` | per-cnv-cluster GMM | high-res MRVI-Leiden majority vote |

> Cells marked **HEAVY** (`normalize/log`, per-donor inferCNV, MRVI Leiden, heatmaps) run on the GPU kernel.

In [ ]:
import sys, gc
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(start)


NB_DIR = _resolve_nb_dir()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
import importlib
import atlas_join_helpers as H
import skin_T_cnv_helpers as C
importlib.reload(H); importlib.reload(C)

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1

V2            = NB_DIR / "data" / "atlas_joint" / "skin_T_tcr_malig_v2.h5ad"  # curated TCR-complete object
GTF           = NB_DIR / "data" / "cache" / "Homo_sapiens.GRCh38.110.chr.gtf.gz"
INTEGRATED_H5 = NB_DIR / "data" / "Integrated_CTCL_skincellatlas_final_portal_tags.h5ad"
UMAP_NPY      = NB_DIR / "data" / "atlas_joint" / "skin_T_umap_u.npy"
OUT_PARQUET   = NB_DIR / "data" / "atlas_joint" / "skin_T_malignancy.parquet"
FIG_DIR       = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

## Step 1 — load the curated TCR-complete skin-T object

Loads `skin_T_tcr_malig_v2.h5ad` (242,959 cells, 34 kept donors): the re-annotated T object whose
TCR malignancy was settled upstream. The ALICE tumor call `tcr_malignant_alice` is carried in as
`tcr_is_malignant` (and `tcr_is_dominant_clone`, so it also drives the benign/diploid reference);
`tcr_is_expanded` / `has_tcr` / `tcr_clone_*` / `cached_malignant` come straight from the file.

In [ ]:
adata = sc.read_h5ad(V2)                       # curated TCR-complete T object (242,959 cells)
# ALICE tumor call is the malignant truth; it also drives benign/reference exclusion downstream
alice = adata.obs["tcr_malignant_alice"].astype(bool).to_numpy()
adata.obs["tcr_is_malignant"]      = alice
adata.obs["tcr_is_dominant_clone"] = alice
# tcr_is_expanded / has_tcr / tcr_clone_size / tcr_clone_id / cached_malignant come from the file
print(adata.shape, "| alice malignant:", int(alice.sum()))
print(adata.obs.groupby("study", observed=True)["has_tcr"].agg(["size", "sum"]))

## Step 2 — carried TCR malignancy (ALICE)

The malignant call is the upstream ALICE tumor annotation (`tcr_malignant_alice`), carried in as
`tcr_is_malignant`. Sanity-check its distribution by `cell_type_T2` (it is CD4-restricted) and by
study.

In [ ]:
# carried ALICE malignancy — distribution by cell_type_T2 and by study
print("tcr_is_malignant by cell_type_T2:\n",
      adata.obs.groupby("cell_type_T2", observed=True)["tcr_is_malignant"].agg(["size", "sum"]))
print("\ntcr_is_malignant by study:\n",
      adata.obs.groupby("study", observed=True)["tcr_is_malignant"].agg(["size", "sum"]))

## Step 3 — inferCNV input prep & reference  · HEAVY (GPU kernel)

The object is **all T cells** (`cell_type_broad == "T"`); the 49-type `cell_type` is sparse, so query / HC-reference are driven off `cell_type_T`. `C.prepare_infercnv_inputs` builds the query and a shared diploid reference, adds genomic positions from the GTF, and returns `acnv, SHARED_REF, CNV_DONORS, HC_DONORS`.

**Query** = every annotated T cell (`cell_type_T != "UNK"`) of the disease-bearing (non-HC) donors; HC donors are held out of the query.

**Reference** (the diploid baseline inferCNV subtracts) — three pooled groups, each used **per donor only if ≥20 cells** are present:

1. **`nonclonal`** — within-donor non-clonal T: `has_tcr` AND not in the donor's dominant TCR clone AND not cached `tumor_cell`. Per-donor internal baseline.
2. **`hc_atlas`** — atlas-internal healthy-control T cells (`disease=="HC"`, `cell_type_T != "UNK"`), sampled up to `N_HC_REF=5000`. Same platform → **preferred**.
3. **`healthy`** — external benign-T from the Integrated CTCL skin cell atlas (`sample_type ∈ {healthy_skin, AD, psoriasis}` AND `cell_type ∈ {Tc, Th, Treg, Tc17_Th17, Tc_IL13_IL22}`), up to `N_HEALTHY_REF=3000`. Cross-platform fallback.

In [ ]:
acnv, SHARED_REF, CNV_DONORS, HC_DONORS = C.prepare_infercnv_inputs(
    adata, GTF, INTEGRATED_H5, SEED, n_hc_ref=5000, n_healthy_ref=3000, use_external=True)

## Step 4 — knobs (resolution & thresholds)

Everything the inferCNV methods are sensitive to, in one place. Change a knob and rerun from Step 5:
- **`CNV_LEIDEN_RES`** — per-cnv-cluster Leiden resolution (the unit methods 3/4 call on). The cache is tagged by it.
- **`CLUSTER_THR_SCALE`** — per-cnv-cluster call threshold (< 1 → more borderline clusters called malignant).
- **`MRVI_LEIDEN_RES` / `MRVI_CLUST_FRAC`** — high-res MRVI-Leiden smoothing shared by methods 2 & 4.

In [ ]:
# >>> KNOBS — change & rerun from Step 5 <<<
# per-cnv-cluster inferCNV build (Step 5):
WINDOW_SIZE       = 250    # window size — smooth 10x dropout
TOPK_FRAC         = 0.10   # focal fraction of windows per cell (CTCL CNVs are focal)
CNV_LEIDEN_RES    = 0.5    # CNV-cluster Leiden resolution (cache is tagged by this)
# per-cnv-cluster call threshold (methods 3 & 4):
CLUSTER_THR_SCALE = 0.8   # < 1 -> more borderline clusters called malignant
# high-res MRVI-Leiden smoothing (methods 2 & 4):
MRVI_LEIDEN_RES   = 2.0    # high resolution -> fine, clone-pure clusters
MRVI_CLUST_FRAC   = 0.6    # cluster malignant if >= this frac of its cells are called
# infercnvpy runtime:
CNV_N_JOBS = 8             # cap workers (one fork per chunk -> OOM if uncapped)
CNV_CHUNK  = 2500
FORCE_CNV  = True          # recompute on the v2 cell set (old cache is 401k); set False after 1st run
# resolution-tagged cache (changing CNV_LEIDEN_RES recomputes into its own file)
CNV_CACHE = NB_DIR / "data" / "atlas_joint" / f"skin_T_cnv_per_cell_res{CNV_LEIDEN_RES:g}.parquet"

## Step 5 — per-donor inferCNV  · HEAVY (GPU kernel, long-running)

Per-donor `cnv.tl.infercnv` vs the combined reference, then a focal per-cell score (mean of each cell's top-`TOPK_FRAC` strongest `|X_cnv|` windows) plus a per-cnv-cluster `cnv_score`. Cached to `CNV_CACHE`; set `FORCE_CNV=True` to recompute.

In [ ]:
C.run_per_donor_infercnv(acnv, SHARED_REF, CNV_DONORS, CNV_CACHE, window=WINDOW_SIZE,
                         topk_frac=TOPK_FRAC, leiden_res=CNV_LEIDEN_RES,
                         n_jobs=CNV_N_JOBS, chunk=CNV_CHUNK, force=FORCE_CNV)

## Step 6 — four malignancy methods (2×2)

`{per-cell GMM, per-cnv-cluster GMM} × {no smoothing, high-res MRVI-Leiden vote}`. The two smoothed methods (2 & 4) reuse **one** MRVI-Leiden clustering. All four calls are propagated onto `adata.obs` and collected in `METHODS`.  · HEAVY (MRVI neighbors + Leiden → GPU kernel)

In [ ]:
# base calls (per-donor GMM crossover + diploid floor)
dip = C.diploid_mask(acnv)
call1, meth1 = C.call_per_donor("cnv_focal_score", dip, CNV_DONORS, acnv.obs, seed=SEED)
acnv.obs["cnv_malignant"]     = call1                 # method 1: per-cell
acnv.obs["cnv_call_method"]   = meth1
call3, _ = C.call_per_donor("cnv_score", dip, CNV_DONORS, acnv.obs, verbose=False,
                            seed=SEED, thr_scale=CLUSTER_THR_SCALE)
acnv.obs["cnv_malig_cluster"] = call3                 # method 3: per-cnv-cluster

# high-res MRVI-Leiden smoothing — one clustering shared by methods 2 & 4
C.compute_mrvi_leiden(acnv, MRVI_LEIDEN_RES, SEED)
acnv.obs["cnv_malig_leiden_mrvi"]  = C.vote_cluster(acnv, "cnv_malignant", MRVI_CLUST_FRAC)
acnv.obs["cnv_malig_cluster_mrvi"] = C.vote_cluster(acnv, "cnv_malig_cluster", MRVI_CLUST_FRAC)

METHODS = {"strat1_cell_gmm":         "cnv_malignant",
           "strat2_cell_mrvi":        "cnv_malig_leiden_mrvi",
           "strat3_cnv_cluster":      "cnv_malig_cluster",
           "strat4_cnv_cluster_mrvi": "cnv_malig_cluster_mrvi"}

# propagate per-cell scores + the 4 calls onto the full object (NaN / "" / False outside the query)
for col in ["cnv_score", "cnv_cell_score", "cnv_focal_score"]:
    adata.obs[col] = acnv.obs[col].reindex(adata.obs_names)
adata.obs["cnv_leiden"]      = acnv.obs["cnv_leiden"].reindex(adata.obs_names).fillna("")
adata.obs["cnv_call_method"] = acnv.obs["cnv_call_method"].reindex(adata.obs_names).fillna("")
for col in METHODS.values():
    adata.obs[col] = acnv.obs[col].reindex(adata.obs_names).fillna(False).astype(bool)

for name, col in METHODS.items():
    print(f"{name:24s} {col:22s} malignant={int(adata.obs[col].sum())}/{adata.n_obs}")

## Step 7 — compare the 4 methods vs TCR

Each inferCNV method benchmarked against the orthogonal TCR-clonality call (`tcr_is_malignant`): precision / recall / F1 / Jaccard on TCR+ query cells, then malignant fraction by study and by donor.

In [ ]:
import matplotlib.pyplot as plt

avail = adata.obs["has_tcr"].to_numpy() & adata.obs["cnv_focal_score"].notna().to_numpy()
agree = C.strategy_agreement(adata, METHODS, "tcr_is_malignant", avail)
print(f"agreement vs TCR on {int(avail.sum())} TCR+ query cells:")
print(agree.to_string())

rename = {"tcr_is_malignant": "TCR", "cnv_malignant": "S1 cell-GMM",
          "cnv_malig_leiden_mrvi": "S2 cell-MRVI", "cnv_malig_cluster": "S3 cnv-cluster",
          "cnv_malig_cluster_mrvi": "S4 cnv-cluster-MRVI"}
frac_cols = ["tcr_is_malignant", *METHODS.values()]
q = adata.obs[adata.obs["donor"].isin(CNV_DONORS)]
for by, fs in [("study", (6, 3)), ("donor", (12, 3))]:
    tbl = q.groupby(by, observed=True)[frac_cols].mean().rename(columns=rename)
    ax = tbl.plot.bar(figsize=fs)
    ax.set_ylabel("malignant fraction"); ax.set_xlabel("")
    ax.set_title(f"CNV methods vs TCR — malignant fraction by {by}")
    plt.tight_layout(); plt.savefig(FIG_DIR / f"skin_T_cnv_methods_frac_by_{by}.png", dpi=150); plt.show()

## Step 8 — UMAP: TCR + the 4 methods

One panel per call on the MRVI u-UMAP. The TCR panel uses `tcr_label` — the **same malignant-clone
scheme as nb16/17** (malignant = ALICE clone, benign = TCR⁺ non-malignant unexpanded, else
unlabeled) — so the malignant set and the benign reference match across notebooks.

In [ ]:
if UMAP_NPY.exists() and np.load(UMAP_NPY).shape[0] == adata.n_obs:
    adata.obsm["X_umap"] = np.load(UMAP_NPY)
else:                                          # stale / missing (cell set changed) -> recompute
    sc.pp.neighbors(adata, use_rep="X_mrvi_u", random_state=SEED)
    sc.tl.umap(adata, random_state=SEED)
    np.save(UMAP_NPY, adata.obsm["X_umap"])
    print("recomputed u-UMAP ->", UMAP_NPY.name)
# tcr_label = identical malignant-clone scheme as nb16/17: malignant = ALICE clone,
# benign = TCR+ non-malignant unexpanded, else unlabeled (so the reference set matches too).
has = adata.obs["has_tcr"].astype(bool).to_numpy()
mal = adata.obs["tcr_is_malignant"].astype(bool).to_numpy()
exp = adata.obs["tcr_is_expanded"].astype(bool).to_numpy()
adata.obs["tcr_label"] = pd.Categorical(
    np.where(mal, "malignant", np.where(has & ~mal & ~exp, "benign", "unlabeled")),
    categories=["malignant", "benign", "unlabeled"])
adata.uns["tcr_label_colors"] = ["#d62728", "#1f77b4", "#dddddd"]   # malignant / benign / unlabeled
for c in METHODS.values():
    adata.obs[c + "_lbl"] = (adata.obs[c].astype(bool)
                             .map({True: "malignant", False: "benign"}).astype("category"))
sc.pl.umap(adata, color=["tcr_label", *[c + "_lbl" for c in METHODS.values()]],
           ncols=2, size=2, wspace=0.3, save="_skin_T_cnv_methods.png")

## Step 9 — chromosome heatmaps for ONE chosen method  · HEAVY (GPU)

Pick a method in `HEATMAP_METHOD`; `C.run_cnv_heatmaps` re-runs inferCNV (keeping `X_cnv`) for the top TCR-burden donors per study and draws a chromosome heatmap with three per-cell strips: the **chosen method's call**, malignant-TCR-clone membership, and clone size.

`ORDER_BY` sets the row grouping: `"cnv_leiden"` (cached Step-5 cluster — the per-cnv-cluster unit, so a cluster-based strip is constant within each block) or `"recompute"` (fresh Leiden on this run's `X_cnv` — visually tighter blocks).

In [ ]:
# >>> pick ONE method to draw chromosome heatmaps for <<<
HEATMAP_METHOD  = "strat1_cell_gmm"   # any key of METHODS
ORDER_BY        = "cnv_leiden"           # "cnv_leiden" (strip uniform per block) | "recompute"
RUN_CNV_HEATMAP = True

if RUN_CNV_HEATMAP:
    C.run_cnv_heatmaps(acnv, SHARED_REF, call_col=METHODS[HEATMAP_METHOD], fig_dir=FIG_DIR,
                       n_per_study=2, window=150, vlim_scale=3.0, cmap="RdBu_r",
                       order_by=ORDER_BY, n_jobs=CNV_N_JOBS, chunk=CNV_CHUNK)
else:
    print("RUN_CNV_HEATMAP=False — skipping the heavy per-donor heatmap recompute")

## Step 10 — combine TCR + CNV & persist

Final CNV call = strategy 3 (`cnv_malig_cluster`). `combined_malignant = TCR ∨ CNV`; `malignant_evidence ∈ {both, tcr_only, cnv_only, none}`. Persist the per-cell annotations (incl. all four method columns) to `OUT_PARQUET`.

In [ ]:
# Final CNV call = strategy 3 (per-cnv-cluster, cnv_malig_cluster).
FINAL_CNV = "cnv_malig_cluster"
tcr_m = adata.obs["tcr_is_malignant"].to_numpy()
cnv_m = adata.obs[FINAL_CNV].to_numpy().astype(bool)
adata.obs["combined_malignant"] = tcr_m | cnv_m
adata.obs["malignant_evidence"] = np.select(
    [tcr_m & cnv_m, tcr_m & ~cnv_m, ~tcr_m & cnv_m],
    ["both", "tcr_only", "cnv_only"], default="none")
print("combined malignant:", int(adata.obs["combined_malignant"].sum()), "/", adata.n_obs)
print("\nevidence:\n", adata.obs["malignant_evidence"].value_counts())

both_avail = adata.obs["has_tcr"].to_numpy() & adata.obs["cnv_focal_score"].notna().to_numpy()
print("\nTCR x CNV crosstab (cells with both signals):")
print(pd.crosstab(adata.obs.loc[both_avail, "tcr_is_malignant"],
                  adata.obs.loc[both_avail, FINAL_CNV], rownames=["tcr"], colnames=["cnv"]))

In [ ]:
keep = ["donor", "study", "disease", "cell_type", "cached_malignant",
        "has_tcr", "tcr_clone_id", "tcr_clone_size", "tcr_is_expanded",
        "tcr_is_dominant_clone", "tcr_is_malignant",
        "cnv_score", "cnv_cell_score", "cnv_focal_score", "cnv_leiden",
        "cnv_call_method", "cnv_malignant",
        "cnv_malig_leiden_mrvi", "cnv_malig_cluster", "cnv_malig_cluster_mrvi",
        "combined_malignant", "malignant_evidence"]
out = adata.obs[keep].copy()
out.index.name = "obs_name"
out.to_parquet(OUT_PARQUET)
print("wrote", OUT_PARQUET, out.shape)
out.head()

In [ ]:
# Cache the 4 per-method CNV malignancy calls (slim, source-prefixed) for cross-notebook comparison (nb18).
METHOD_COLS = {
    "cnv_malignant":          "cnvT_cell_gmm",
    "cnv_malig_leiden_mrvi":  "cnvT_cell_mrvi",
    "cnv_malig_cluster":      "cnvT_cnvcluster",
    "cnv_malig_cluster_mrvi": "cnvT_cnvcluster_mrvi",
}
meth = adata.obs[list(METHOD_COLS)].rename(columns=METHOD_COLS).astype(bool)
meth.index.name = "obs_name"
METHODS_PARQUET = NB_DIR / "data" / "atlas_joint" / "skin_T_malignancy_methods.parquet"
meth.to_parquet(METHODS_PARQUET)
print("wrote", METHODS_PARQUET, meth.shape, {c: int(meth[c].sum()) for c in meth.columns})

## Step 11 — TCR↔CNV quality metrics

For each method, restricted to TCR+ cells, pooled agreement with the TCR call:
**sensitivity** = of TCR-malignant cells, fraction called CNV-malignant; **specificity** =
of TCR-non-malignant cells, fraction called CNV-non-malignant (with raw counts).

In [ ]:
# Step 11 — TCR<->CNV quality metrics (TCR+ cells, per method)
avail = adata.obs["has_tcr"].to_numpy() & adata.obs["cnv_focal_score"].notna().to_numpy()
quality = C.tcr_cnv_quality(adata, METHODS, "tcr_is_malignant", avail)
print(f"TCR<->CNV quality on {int(avail.sum())} TCR+ cells:")
print(quality.to_string())